# ATS Scorer - Cross-Encoder Semantic Similarity

This notebook demonstrates how we use a Transformer Cross-Encoder model (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to score candidate resumes against target job descriptions. Cross-encoders feed the query and document together into the transformer attention layers, yielding superior semantic matching accuracy compared to traditional keyword trackers.

In [1]:
import os
import torch
from sentence_transformers import CrossEncoder
import pandas as pd
import numpy as np

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. Load Pre-trained Cross-Encoder Model

We initialize the model from Hugging Face. The output of a Cross-Encoder is a continuous score representing query-document relevance.

In [2]:
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
model = CrossEncoder(model_name)
print("Cross-Encoder model loaded successfully!")

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vansh Agrawal\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Cross-Encoder model loaded successfully!


## 2. Evaluate Resume-JD Semantic Alignment

In [3]:
jd = "We are seeking a Backend Engineer with expert Python, FastAPI, Postgres, and Docker skills."

candidate_resumes = [
    {"name": "Alice", "text": "Backend developer with Python, FastAPI microservices experience, Postgres schema design, and Docker deployment.", "profile": "Perfect Match"},
    {"name": "Bob", "text": "Frontend specialist skilled in React, HTML, CSS, javascript, UI design systems, and web accessibility.", "profile": "Unrelated Stack"},
    {"name": "Charlie", "text": "Software engineer with general coding experience in Python, simple databases, and basic server tools.", "profile": "Partial Match"}
]

# Form pairs for Cross-Encoder inference
pairs = [(jd, r["text"]) for r in candidate_resumes]
scores = model.predict(pairs)

results = []
for idx, score in enumerate(scores):
    # Calibrate sigmoid score into a 0-100 percentage metric
    calibrated_score = 100 / (1 + np.exp(-score)) 
    results.append({
        "Candidate": candidate_resumes[idx]["name"],
        "Expected Profile": candidate_resumes[idx]["profile"],
        "Raw Output Score": score,
        "Calibrated ATS Score (%)": int(calibrated_score)
    })

df_results = pd.DataFrame(results).sort_values(by="Calibrated ATS Score (%)", ascending=False)
df_results

,Candidate,Expected Profile,Raw Output Score,Calibrated ATS Score (%)
0,Alice,Perfect Match,5.047821,99
1,Bob,Unrelated Stack,-2.582140,7
2,Charlie,Partial Match,-3.591120,2
